# Socceraction

## Libraries and Constants

In [7]:
# Import libraries
import pandas as pd
import socceraction.spadl as spadl
import socceraction.vaep.features as fs
import socceraction.vaep.labels as lab

from pathlib import Path
from tqdm import tqdm

In [8]:
# Ignore warnings
import warnings

warnings.filterwarnings(action="ignore", category=FutureWarning)
warnings.filterwarnings(action="ignore", message="Inferred xy_fidelity_version=2.", category=UserWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [9]:
# Define the data directory
DATA_DIR = Path("../data")
SPADL_DIR = Path("data")

SPADL_H5 = SPADL_DIR / "spadl-statsbomb.h5"
FEATURES_H5 = SPADL_DIR / "features.h5"
LABELS_H5 = SPADL_DIR / "labels.h5"
PREDICTIONS_H5 = SPADL_DIR / "predictions.h5"

## VAEP

In [10]:
games_df = pd.read_hdf(SPADL_H5, "games")
print(f"Number of games: {len(games_df)}")

Number of games: 380


In [11]:
xfns = [
    fs.actiontype,
    fs.actiontype_onehot,
    fs.bodypart,
    fs.bodypart_onehot,
    fs.result,
    fs.result_onehot,
    fs.goalscore,
    fs.startlocation,
    fs.endlocation,
    fs.movement,
    fs.space_delta,
    fs.startpolar,
    fs.endpolar,
    fs.team,
    fs.time,
    fs.time_delta,
]

with pd.HDFStore(SPADL_H5) as spadlstore, pd.HDFStore(FEATURES_H5) as featurestore:
    for game in tqdm(list(games_df.itertuples()), desc=f"Generating and storing features in {FEATURES_H5}"):
        actions = spadlstore[f"actions/game_{game.game_id}"]
        gamestates = fs.gamestates(spadl.add_names(actions), 3)
        gamestates = fs.play_left_to_right(gamestates, game.home_team_id)

        X = pd.concat([fn(gamestates) for fn in xfns], axis=1)
        featurestore.put(f"game_{game.game_id}", X, format="table")

Generating and storing features in data\features.h5: 100%|██████████| 380/380 [02:20<00:00,  2.70it/s]


In [ ]:
yfns = [lab.scores, lab.concedes, lab.goal_from_shot]

with pd.HDFStore(SPADL_H5) as spadlstore, pd.HDFStore(LABELS_H5) as labelstore:
    for game in tqdm(list(games_df.itertuples()), desc=f"Computing and storing labels in {LABELS_H5}"):
        actions = spadlstore[f"actions/game_{game.game_id}"]
        Y = pd.concat([fn(spadl.add_names(actions)) for fn in yfns], axis=1)
        labelstore.put(f"game_{game.game_id}", Y, format="table")

Computing and storing labels in data\labels.h5: 100%|██████████| 380/380 [00:35<00:00, 10.74it/s]
